# 🔬 Python Internals — Memory & Mutability
### *id(), is vs ==, shallow/deep copy, interning, GIL*

---

> **Mental Model First:**
> Every Python object is a box sitting somewhere in memory. The box has a label
> (its address), a type stamp, a reference count, and its value. Variables are
> sticky notes — they point to boxes, not boxes themselves. Two sticky notes can
> point to the same box.

---

## 📋 Table of Contents

| # | Section |
|---|---------|
| 1 | [Objects, References & id()](#1) |
| 2 | [is vs == — Identity vs Equality](#2) |
| 3 | [Mutable vs Immutable](#3) |
| 4 | [Shallow vs Deep Copy](#4) |
| 5 | [Integer Interning & String Interning](#5) |
| 6 | [Reference Counting & Garbage Collection](#6) |
| 7 | [The GIL — Global Interpreter Lock](#7) |
| 8 | [Decision Map & Cheat Sheet](#8) |


<a id='1'></a>

## 1. Objects, References & id()

---

```
PYTHON MEMORY MODEL

  Variable names are just labels (references).
  The object lives on the heap.

  a = [1, 2, 3]          a ──────────────► [1, 2, 3]  (list object, refcount=1)
  b = a                  b ──────────────► [1, 2, 3]  (same object, refcount=2)
  c = [1, 2, 3]          c ──────────────► [1, 2, 3]  (new object, refcount=1)

  id(a) == id(b)   → True   (same box, same address)
  id(a) == id(c)   → False  (different boxes, equal contents)
  a == c           → True   (equal contents)
  a is c           → False  (different objects)

  id() returns the memory address (CPython implementation).
  Two live objects NEVER share an id at the same time.
```


In [ ]:
# id() reveals the memory address of an object
a = [1, 2, 3]
b = a           # b points to SAME object — no copy made
c = [1, 2, 3]   # c points to NEW object with equal contents

print(f"id(a) = {id(a)}")
print(f"id(b) = {id(b)}")
print(f"id(c) = {id(c)}")
print(f"id(a) == id(b): {id(a) == id(b)}")   # True — same address
print(f"id(a) == id(c): {id(a) == id(c)}")   # False — different objects

# Mutation through a is visible through b — they share the box
a.append(4)
print(f"After a.append(4):")
print(f"  a = {a}")
print(f"  b = {b}")   # b sees the change — same box
print(f"  c = {c}")   # c unchanged — its own box

# But c still equals a element-wise
print(f"a == c: {a == c}")   # False now (a has 4 extra)
c.append(4)
print(f"After c.append(4), a == c: {a == c}")   # True again
print("id() demo complete.")


<a id='2'></a>

## 2. is vs == — Identity vs Equality

---

```
is    → checks memory address (id(a) == id(b))
==    → checks value equality (__eq__ method)

RULES:
  Use `is` ONLY for:          Use `==` for:
  ─────────────────────────   ─────────────────────────
  x is None                   x == None  (works but bad)
  x is True / x is False      value comparison of any type
  x is not None               x != something

NEVER use `is` to compare:
  ❌  if x is "hello":       # might work by accident (interning)
  ❌  if x is 1000:          # FAILS for large ints
  ✅  if x == "hello":
  ✅  if x == 1000:

WHY `is None` is correct:
  None is a singleton — there is exactly ONE None object in Python.
  `is None` checks "is this the None object" (address check).
  `== None` calls __eq__ which could be overridden by a library class.
```


In [ ]:
# is vs == demonstration
a = [1, 2, 3]
b = a
c = [1, 2, 3]

print(f"a is b: {a is b}")   # True — same object
print(f"a is c: {a is c}")   # False — different objects
print(f"a == c: {a == c}")   # True — equal contents

# None comparison — always use `is`
x = None
print(f"x is None: {x is None}")    # correct
print(f"x == None: {x == None}")    # works but fragile

# ── Integer caching surprise ────────────────────────────────────────────────
# CPython caches small integers (-5 to 256) — they ARE the same object
small_a = 100
small_b = 100
print(f"100 is 100: {small_a is small_b}")   # True (cached)

large_a = 1000
large_b = 1000
print(f"1000 is 1000: {large_a is large_b}")  # False (not cached!)
print(f"1000 == 1000: {large_a == large_b}")  # True (equal value)

# ── Boolean is check ─────────────────────────────────────────────────────────
# True and False are singletons
flag = True
print(f"flag is True: {flag is True}")   # True — singletons

# ── Class with __eq__ override ────────────────────────────────────────────────
class AlwaysEqual:
    def __eq__(self, other): return True   # lies about equality

obj = AlwaysEqual()
print(f"obj == None: {obj == None}")   # True — __eq__ overridden
print(f"obj is None: {obj is None}")   # False — not the None object
print("is vs == demo complete.")


<a id='3'></a>

## 3. Mutable vs Immutable

---

```
IMMUTABLE (cannot change in-place)     MUTABLE (can change in-place)
─────────────────────────────────────  ─────────────────────────────────────
int, float, bool, complex              list
str                                    dict
tuple                                  set
frozenset                              bytearray
bytes                                  custom objects (usually)
NoneType

WHY THIS MATTERS:
  Immutable → safe to share. No one can change it under you.
  Mutable   → beware aliasing. Two names, one object, shared state.

REASSIGNMENT vs MUTATION:
  x = 5       # x points to int object 5
  x = x + 1   # x now points to NEW int object 6. Old 5 unchanged.

  lst = [1,2]
  lst.append(3)   # MUTATES the object in-place. Any alias sees change.
  lst = [4,5]     # REASSIGNMENT. lst now points to new list. Old [1,2,3] unchanged.

TUPLE GOTCHA:
  t = (1, [2, 3], 4)
  t[1].append(99)   # OK! tuple is immutable, but it CONTAINS a mutable list
  print(t)          # (1, [2, 3, 99], 4)
  t[0] = 99         # TypeError — can't change the tuple's slots
```


In [ ]:
# Mutable vs immutable in action

# ── Strings are immutable ─────────────────────────────────────────────────────
s = "hello"
print(f"id before: {id(s)}")
s += " world"   # creates a NEW string — does NOT modify the old one
print(f"id after:  {id(s)}")   # different address

# ── Lists are mutable ─────────────────────────────────────────────────────────
lst = [1, 2, 3]
print(f"id before: {id(lst)}")
lst.append(4)           # modifies IN PLACE — same object
print(f"id after:  {id(lst)}")   # SAME address
print(f"lst = {lst}")

# ── Tuple immutability with mutable contents ──────────────────────────────────
t = (1, [2, 3], 4)
t[1].append(99)          # list inside tuple is still mutable!
print(f"t = {t}")        # (1, [2, 3, 99], 4)
try:
    t[0] = 99            # tuple slots immutable
except TypeError as e:
    print(f"t[0] = 99 → {e}")

# ── Default argument gotcha (classic interview trap) ─────────────────────────
def bad_append(item, lst=[]):    # lst=[] created ONCE at function definition
    lst.append(item)
    return lst

print(bad_append(1))   # [1]
print(bad_append(2))   # [1, 2]  ← surprise! shared mutable default
print(bad_append(3))   # [1, 2, 3]

def good_append(item, lst=None):   # None is immutable singleton
    if lst is None:
        lst = []       # fresh list each call
    lst.append(item)
    return lst

print(good_append(1))   # [1]
print(good_append(2))   # [2]  ← correct, independent lists
print("Mutable/immutable demo complete.")


<a id='4'></a>

## 4. Shallow vs Deep Copy

---

```
ASSIGNMENT (no copy):
  b = a              b and a point to SAME object

SHALLOW COPY:
  b = a[:]           new container, same inner objects
  b = list(a)        same
  b = copy.copy(a)   same

  a = [[1, 2], [3, 4]]
  b = a[:]           # new outer list
  b[0].append(99)    # modifies inner list — shared between a and b!
  print(a)           # [[1, 2, 99], [3, 4]]  ← a is affected

DEEP COPY:
  b = copy.deepcopy(a)   new container AND all inner objects recursively

  b[0].append(99)    # only b is affected — no sharing
  print(a)           # [[1, 2], [3, 4]]  ← a unchanged

WHEN TO USE WHAT:
  Assignment   → when you WANT aliasing (intentional shared state)
  Shallow copy → flat structures, or when inner objects won't be modified
  Deep copy    → nested mutable structures, graph traversal state, etc.
  Deep copy is SLOW for large objects — avoid in hot paths.
```


In [ ]:
import copy

original = [[1, 2], [3, 4], [5, 6]]

# ── Assignment — no copy ──────────────────────────────────────────────────────
alias = original
alias[0].append(99)
print(f"After alias[0].append(99):")
print(f"  original = {original}")   # changed! same object
alias[0].pop()          # undo for next demo

# ── Shallow copy ─────────────────────────────────────────────────────────────
shallow = original[:]        # new outer list, same inner lists
shallow[0].append(99)
print(f"After shallow[0].append(99):")
print(f"  original = {original}")   # inner list SHARED — original changed!
print(f"  shallow  = {shallow}")
shallow[0].pop()             # undo

# New outer list — adding a new inner list doesn't affect original
shallow.append([7, 8])
print(f"After shallow.append([7,8]):")
print(f"  original = {original}")   # unchanged — new slot in shallow only
print(f"  shallow  = {shallow}")

# ── Deep copy ─────────────────────────────────────────────────────────────────
deep = copy.deepcopy(original)
deep[0].append(99)
print(f"After deep[0].append(99):")
print(f"  original = {original}")   # UNCHANGED — completely independent
print(f"  deep     = {deep}")

# ── id() proof ────────────────────────────────────────────────────────────────
print(f"id(original[0]) == id(shallow[0]): {id(original[0]) == id(shallow[0])}")  # True
print(f"id(original[0]) == id(deep[0]):    {id(original[0]) == id(deep[0])}")     # False
print("Copy demo complete.")


<a id='5'></a>

## 5. Integer Interning & String Interning

---

```
INTERNING = reusing the same object for equal values.
CPython interns certain objects for performance.

INTEGER INTERNING (CPython):
  Range: -5 to 256 (inclusive)
  These are pre-created at startup — always same object.

  a = 100; b = 100; a is b → True   (both point to cached int 100)
  a = 1000; b = 1000; a is b → False  (two separate objects)

STRING INTERNING:
  CPython automatically interns strings that look like identifiers
  (only letters, digits, underscores, starting with letter/underscore).

  a = "hello"; b = "hello"; a is b → True   (interned — same object)
  a = "hello world"; b = "hello world"; a is b → MAYBE True  (can vary)
  a = "hello!"; b = "hello!"; a is b → False  (non-identifier chars)

  sys.intern(s) forces interning of any string.

WHY THIS MATTERS FOR LC:
  ❌  Never use `is` to compare string values in production code
  ✅  Always use == for value comparison
  The interning behavior is CPython implementation detail — can change.

NONE / TRUE / FALSE:
  These are always singletons — `is` is correct and preferred.
```


In [ ]:
import sys

# ── Integer interning: -5 to 256 ─────────────────────────────────────────────
print("Integer interning:")
for n in [-6, -5, 0, 100, 256, 257, 1000]:
    a = n; b = n
    # Force separate references by using exec
    exec(f"x = {n}; y = {n}; print(f'  {{n:>6}}: x is y = {{x is y}}')")

# ── String interning ─────────────────────────────────────────────────────────
print("String interning:")
s1 = "hello"
s2 = "hello"
s3 = "hello world"
s4 = "hello world"
s5 = "hello!"
s6 = "hello!"

# Build strings at runtime to avoid compile-time folding
s3_rt = "hello" + " " + "world"
s4_rt = "hello" + " " + "world"

print(f'  "hello" is "hello": {s1 is s2}')             # True — identifier-like
print(f'  "hello world" is "hello world": {s3 is s4}')  # True — compile folded
print(f'  runtime "hello world" is same: {s3_rt is s4_rt}')  # might be False
print(f'  "hello!" is "hello!": {s5 is s6}')            # False — contains !

# sys.intern forces interning
a = sys.intern(s3_rt)
b = sys.intern(s4_rt)
print(f'  after sys.intern: {a is b}')   # True — forced same object

# ── Practical: always use == not is for strings ───────────────────────────────
user_input = "".join(["h","e","l","l","o"])   # runtime construction
expected = "hello"
print(f'  user_input is expected: {user_input is expected}')  # might be False
print(f'  user_input == expected: {user_input == expected}')  # True — always correct
print("Interning demo complete.")


<a id='6'></a>

## 6. Reference Counting & Garbage Collection

---

```
CPython MEMORY MANAGEMENT:
  Every object has a reference count.
  When refcount drops to 0 → object freed immediately.

  a = [1,2,3]    # refcount = 1
  b = a          # refcount = 2
  del a          # refcount = 1
  del b          # refcount = 0 → freed

CYCLIC GARBAGE:
  Refcounting can't handle cycles:
  a = []; b = []; a.append(b); b.append(a)
  del a; del b   # both still refcount=1 (each points to other)
  → Memory leak!

  CPython has a CYCLE COLLECTOR (gc module) that detects and cleans cycles.
  Runs periodically. Can be triggered with gc.collect().

WEAK REFERENCES (weakref module):
  Don't increment refcount — let the GC collect the object.
  Used for caches — you want to cache but not prevent collection.

  import weakref
  cache = weakref.WeakValueDictionary()

PRACTICAL FOR INTERVIEWS:
  - Don't worry about manual memory management in Python
  - Large objects: del them explicitly if memory matters (e.g., big DP table)
  - Circular refs in your own classes: use __del__ carefully or weakref
```


In [ ]:
import sys, gc

# ── Reference count tracking ────────────────────────────────────────────────
lst = [1, 2, 3]
# sys.getrefcount returns count + 1 (the getrefcount call itself adds 1)
print(f"refcount(lst) after creation: {sys.getrefcount(lst) - 1}")   # 1

alias = lst
print(f"refcount after alias = lst:   {sys.getrefcount(lst) - 1}")   # 2

another = lst
print(f"refcount after another = lst: {sys.getrefcount(lst) - 1}")   # 3

del alias
print(f"refcount after del alias:     {sys.getrefcount(lst) - 1}")   # 2

del another
print(f"refcount after del another:   {sys.getrefcount(lst) - 1}")   # 1

# ── Cyclic garbage ────────────────────────────────────────────────────────────
gc.collect()   # clean up any existing cycles
before = gc.get_count()

# Create a cycle: a → b → a
a = {}
b = {}
a['ref'] = b   # a points to b
b['ref'] = a   # b points to a
del a, del b   # drop Python-level names — but objects still reference each other

after_del = gc.get_count()
collected = gc.collect()   # force cycle collection
after_gc = gc.get_count()

print(f"Cycle collected: {collected} objects freed by GC")
print(f"gc counts before: {before}, after del: {after_del}, after gc: {after_gc}")

# ── Memory tip: explicit del for large objects ────────────────────────────────
import array as arr_mod

big_list = list(range(1_000_000))   # ~8MB
print(f"Before del: big_list exists, refcount = {sys.getrefcount(big_list) - 1}")

del big_list   # immediately freed (refcount → 0, no cycles)
try:
    _ = big_list
except NameError:
    print("After del: big_list freed — NameError confirms it's gone")

print("Reference counting demo complete.")


<a id='7'></a>

## 7. The GIL — Global Interpreter Lock

---

```
WHAT IS THE GIL?
  A mutex (lock) inside CPython that allows only ONE thread to
  execute Python bytecode at a time.

  Thread 1: ──[run]──[run]──[yield GIL]──[wait]──────────────[run]──
  Thread 2: ──[wait]──────────[get GIL]──[run]──[run]──[yield GIL]──

CONSEQUENCE:
  Python threads do NOT run truly in parallel for CPU-bound work.
  For CPU-bound tasks: threading gives NO speedup (often slower).
  For I/O-bound tasks: threading DOES help — threads release GIL while waiting.

WHEN EACH APPLIES:
  ──────────────────────────────────────────────────────────────
  CPU-bound (matrix math, sorting, hashing)
  → Use multiprocessing or concurrent.futures.ProcessPoolExecutor
  → Each process has its OWN Python interpreter + GIL

  I/O-bound (network, disk, sleep, database queries)
  → Use threading or asyncio
  → GIL released during I/O wait → other threads run

GIL IN INTERVIEWS:
  Common question: "Python has threads, why doesn't X run faster?"
  Answer: GIL serializes CPU work. Use multiprocessing for CPU parallelism.

PYTHON 3.13+:
  Experimental "no-GIL" build available (PEP 703).
  Not yet default. Watch this space.
```


In [ ]:
import threading, time, concurrent.futures

# ── CPU-bound: threads vs processes comparison ──────────────────────────────
def cpu_work(n):
    # Pure CPU work — counting
    count = 0
    for _ in range(n):
        count += 1
    return count

N = 5_000_000

# Sequential
start = time.perf_counter()
cpu_work(N)
cpu_work(N)
t_seq = time.perf_counter() - start

# Two threads (GIL prevents true parallelism)
start = time.perf_counter()
t1 = threading.Thread(target=cpu_work, args=(N,))
t2 = threading.Thread(target=cpu_work, args=(N,))
t1.start(); t2.start()
t1.join(); t2.join()
t_threads = time.perf_counter() - start

print(f"Sequential:  {t_seq*1000:.1f}ms")
print(f"2 Threads:   {t_threads*1000:.1f}ms  (GIL: not much faster / possibly slower)")

# Multiprocessing bypasses GIL
with concurrent.futures.ProcessPoolExecutor(max_workers=2) as pool:
    start = time.perf_counter()
    futures = [pool.submit(cpu_work, N), pool.submit(cpu_work, N)]
    results = [f.result() for f in futures]
    t_proc = time.perf_counter() - start

print(f"2 Processes: {t_proc*1000:.1f}ms  (true parallelism, ~2x faster)")

# ── I/O-bound: threads DO help ───────────────────────────────────────────────
def io_work():
    time.sleep(0.1)   # simulates I/O — GIL released during sleep

# Sequential I/O
start = time.perf_counter()
io_work(); io_work()
t_seq_io = time.perf_counter() - start

# Threaded I/O
start = time.perf_counter()
t1 = threading.Thread(target=io_work)
t2 = threading.Thread(target=io_work)
t1.start(); t2.start()
t1.join(); t2.join()
t_thr_io = time.perf_counter() - start

print(f"Sequential I/O: {t_seq_io*1000:.1f}ms")
print(f"Threaded I/O:   {t_thr_io*1000:.1f}ms  (GIL released — ~2x faster)")
print("GIL demo complete.")


<a id='8'></a>

## 8. Decision Map & Cheat Sheet

---

```
QUESTION                               ANSWER
──────────────────────────────────────────────────────────────────
"Is this the same object?"            Use `is` (only for None/True/False)
"Do these have the same value?"       Use ==
"Why did my function change my list?" Aliasing — assignment is not a copy
"How to avoid aliasing?"              b = a[:] (shallow) or copy.deepcopy(a)
"Why does 1000 is 1000 → False?"      Integer interning only up to 256
"Why don't my threads speed things?"  GIL — use multiprocessing for CPU work
"Memory leak in my linked list?"      Cyclic refs — gc.collect() or weakref
```

**When to use each:**

| Situation | Tool |
|-----------|------|
| Compare to None | `x is None` |
| Compare values | `x == y` |
| Prevent aliasing (flat) | `lst[:]` or `list(lst)` |
| Prevent aliasing (nested) | `copy.deepcopy(obj)` |
| CPU parallelism | `multiprocessing` |
| I/O parallelism | `threading` or `asyncio` |
| Forced string dedup | `sys.intern(s)` |

**Gotchas:**

```
❌  b = a   then mutating b also mutates a (aliasing)
❌  def f(lst=[]):   mutable default — shared across calls
❌  1000 is 1000   may be False — use ==
❌  "hello!" is "hello!"   may be False — use ==
❌  threads for CPU work — GIL blocks parallel execution
✅  b = a[:]   shallow copy for flat lists
✅  copy.deepcopy()   for nested mutable structures
✅  def f(lst=None): if lst is None: lst = []
✅  multiprocessing for CPU-bound parallel work
```


```
               🔬 PYTHON INTERNALS MAP

               VARIABLES
               └─ names pointing to objects (not boxes)
               └─ assignment = redirect pointer

               IDENTITY vs EQUALITY
               ├─ is → same object (same id())
               └─ == → same value (__eq__)

               MUTABILITY
               ├─ Immutable: int, str, tuple, frozenset
               │  └─ "changes" create NEW objects
               └─ Mutable: list, dict, set, custom objects
                  └─ changes modify IN PLACE → aliasing risk

               COPYING
               ├─ Assignment: no copy, shared alias
               ├─ Shallow copy a[:]: new container, shared inner objects
               └─ copy.deepcopy(): fully independent clone

               INTERNING
               ├─ Integers -5..256: always same object
               ├─ Identifier-like strings: usually interned
               └─ Rule: always == for value, `is` only for None/True/False

               MEMORY MANAGEMENT
               ├─ Refcounting: freed when count → 0
               ├─ Cycle collector: handles circular refs
               └─ GIL: one thread runs bytecode at a time
                  ├─ CPU-bound → multiprocessing
                  └─ I/O-bound → threading / asyncio

---
*End of Python Internals Guide — Sean Edition*
```
